<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/alignnfffd_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [ALIGNN-FF fd_path](https://github.com/usnistgov/alignn) prediction for JARVIS-Leaderboard

In [ ]:
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .


In [ ]:
!pip install dgl==1.0.1+cu117 -f https://data.dgl.ai/wheels/cu117/repo.html
!pip install alignn


In [ ]:
!pwd

In [ ]:
# %%time
# !pip install --pre dgl -f https://data.dgl.ai/wheels/cu118/repo.html
# !pip install --pre dglgo -f https://data.dgl.ai/wheels-test/repo.html
# !pip install -q alignn

In [ ]:
import zipfile
import json
import glob
import pandas as pd
import numpy as np
from jarvis.core.atoms import Atoms
import os
from ase.stress import full_3x3_to_voigt_6_stress, voigt_6_to_full_3x3_stress
from alignn.ff.ff import AlignnAtomwiseCalculator, default_path, ForceField, fd_path
model_path = fd_path()


calc = AlignnAtomwiseCalculator(
    path=model_path,
    force_mult_natoms=False,
    force_multiplier=25,
    stress_wt=-4800,
)
# export CUDA_VISIBLE_DEVICES=""
# wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip


def get_alignn_forces(atoms):
    energy = 0.0
    forces = np.zeros((atoms.num_atoms, 3))
    stress = np.zeros((3, 3))
    # try:
    ase_atoms = atoms.ase_converter()
    ase_atoms.calc = calc  # M3GNetCalculator(potential=potential)
    forces = np.array(ase_atoms.get_forces())
    energy = ase_atoms.get_potential_energy()
    stress = voigt_6_to_full_3x3_stress(ase_atoms.get_stress())
    # except:
    #  print ('Failed for',atoms)
    #  pass
    return energy, forces, stress

In [ ]:
if not os.path.exists('jarvis_leaderboard/contributions/alignnff_fd_test'):
    os.makedirs('jarvis_leaderboard/contributions/alignnff_fd_test')
os.chdir('jarvis_leaderboard/contributions/alignnff_fd_test')

In [ ]:
import os
os.chdir('/content/jarvis_leaderboard/jarvis_leaderboard/contributions/alignnff_fd_test')
cmd='cp ../alignnff_fd/metadata.json .'
os.system(cmd)

In [ ]:
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
df = pd.DataFrame(json.loads(zipfile.ZipFile('mlearn.json.zip').read('mlearn.json')))
print (df)
for i in glob.glob('../../benchmarks/AI/MLFF/*energy*.zip'):
 if 'mlearn' in i:
    fname_e='AI-MLFF-energy-'+i.split('/')[-1].split('_energy.json.zip')[0]+'-test-mae.csv'
    fname_f='AI-MLFF-forces-'+i.split('/')[-1].split('_energy.json.zip')[0]+'-test-multimae.csv'
    #fname_s='AI-MLFF-stresses-'+i.split('/')[-1].split('_energy.json.zip')[0]+'-test-multimae.csv'
    f_e=open(fname_e,'w')
    f_f=open(fname_f,'w')
    #f_s=open(fname_s,'w')

    f_e.write('id,prediction\n')
    f_f.write('id,prediction\n')
    #f_s.write('id,prediction\n')


    print (i)
    dat=json.loads(zipfile.ZipFile(i).read(i.split('/')[-1].split('.zip')[0]))
    print(dat['test'])
    for key,val in dat['test'].items():
        entry = df[df['jid']==key]
        atoms=Atoms.from_dict(entry.atoms.values[0])
        #print(key,val,df[df['jid']==key],atoms)
        energy,forces,stress=get_alignn_forces(atoms)
        print (key,val,atoms.num_atoms)
        line=key+','+str(atoms.num_atoms)+'\n'
        f_e.write(line)
        line=key+','+str(';'.join(map(str,np.array(forces).flatten())))+'\n'
        f_f.write(line)
        #line=key+','+str(';'.join(map(str,np.array(stress).flatten())))+'\n'
        #f_s.write(line)
    f_e.close()
    f_f.close()
    #f_s.close()
    cmd = 'zip '+fname_e+'.zip '+fname_e
    os.system(cmd)
    cmd = 'zip '+fname_f+'.zip '+fname_f
    os.system(cmd)
    #cmd = 'zip '+fname_s+'.zip '+fname_s
    #os.system(cmd)
    cmd='rm '+fname_e
    os.system(cmd)
    cmd='rm '+fname_f
    os.system(cmd)
    #cmd='rm '+fname_s
    #os.system(cmd)
    #break

In [ ]:
os.chdir('/content')

# Restart runtime here

In [ ]:
from jarvis_leaderboard.rebuild import get_metric_value,get_results
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
names,vals=get_results(bench_name='AI-MLFF-forces-mlearn_Cu-test-multimae.csv.zip')
plt.bar(np.arange(len(vals)),vals,color=(0.2, 0.4, 0.6, 0.6),edgecolor='blue')
plt.xticks(np.arange(len(vals)),names,rotation=90)
plt.ylabel('MAE (eV/A)')

In [ ]:
from jarvis_leaderboard.rebuild import get_metric_value,get_results
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
names,vals=get_results(bench_name='AI-MLFF-energy-mlearn_Cu-test-mae.csv.zip')
plt.bar(np.arange(len(vals)),vals,color=(0.2, 0.4, 0.6, 0.6),edgecolor='blue')
plt.xticks(np.arange(len(vals)),names,rotation=90)
plt.ylabel('MAE')
#Note ALIGNN-FF trained on OptB88vdW functional data, not PBE
plt.ylim([0,5])

In [ ]:
for i,j in zip(names,vals):
  print(i,j)

In [ ]:
pip freeze